# DATAFLOW 2026 - User Behavior Prediction
## Part 2: Feature Engineering

---

### Data Policy — STRICT RULE (must never be violated)

| Rule | Detail |
|------|--------|
| `SequencePreprocessor.fit()` | **Only on `train_sequences`** — never on val or test |
| `FeaturePipeline.fit()` | **Only on `train_sequences`** — never on val or test |
| `TargetEncoder.fit()` | **Only on `Y_train`** — never on val labels |
| `Y_val` | Transformed (not fitted) — used only for offline evaluation in notebook 03 |
| `merge_train_val()` | **NEVER called** |
| Dense vocab mapping | Built from train tokens only; unknown val/test tokens map to 0 (padding) |

---

### Notebook Contents
1. Sequence Preprocessing (SequencePreprocessor — dense vocab remapping + padding)
2. TF-IDF Features
3. N-gram Features
4. Statistical + Histogram Features
5. Full Feature Pipeline (for XGBoost / LightGBM)
6. Target Encoding

In [1]:
import sys
sys.path.append('..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

from src.data import DataLoader
from src.data.preprocessor import (
    SequencePreprocessor, 
    TargetEncoder, 
    extract_sequences_from_df
)
from src.features import (
    SequenceFeatureExtractor,
    StatisticalFeatureExtractor,
    FeaturePipeline
)
from src.utils import set_seed, load_config

# Create output directories
Path('../outputs/figures').mkdir(parents=True, exist_ok=True)
Path('../outputs/models').mkdir(parents=True, exist_ok=True)
Path('../outputs/submissions').mkdir(parents=True, exist_ok=True)

set_seed(42)
%matplotlib inline

In [2]:
# Load data
config = load_config('../configs/config.yaml')
loader = DataLoader('../data')

X_train, Y_train = loader.load_train()
X_val, Y_val = loader.load_val()

train_ids, train_sequences = extract_sequences_from_df(X_train)
val_ids, val_sequences = extract_sequences_from_df(X_val)

print(f"Training sequences: {len(train_sequences)}")
print(f"Validation sequences: {len(val_sequences)}")

2026-02-28 11:06:37,033 - src.data.loader - INFO - Loaded X_train.csv: (51000, 38)
2026-02-28 11:06:37,065 - src.data.loader - INFO - Loaded Y_train.csv: (51000, 7)
2026-02-28 11:06:37,066 - src.data.loader - INFO - Loaded training data: X=51000, Y=51000
2026-02-28 11:06:37,086 - src.data.loader - INFO - Loaded X_val.csv: (7200, 38)
2026-02-28 11:06:37,093 - src.data.loader - INFO - Loaded Y_val.csv: (7200, 7)
2026-02-28 11:06:37,093 - src.data.loader - INFO - Loaded validation data: X=7200, Y=7200


Training sequences: 51000
Validation sequences: 7200


## 1. Sequence Preprocessing

In [3]:
# Create sequence preprocessor
preprocessor = SequencePreprocessor(
    max_length=config['preprocessing']['max_sequence_length'],
    padding_strategy=config['preprocessing']['padding_strategy'],
    truncation_strategy=config['preprocessing']['truncation_strategy'],
)

# Fit and transform
X_train_padded = preprocessor.fit_transform(train_sequences)
X_val_padded = preprocessor.transform(val_sequences)

print(f"Padded train shape: {X_train_padded.shape}")
print(f"Padded val shape: {X_val_padded.shape}")
print(f"Vocabulary size: {preprocessor.vocab_size}")

2026-02-28 11:06:52,969 - src.data.preprocessor - INFO - Fitted preprocessor: 254 unique tokens → dense vocab_size=255 (was 24439 with raw IDs)


Padded train shape: (51000, 64)
Padded val shape: (7200, 64)
Vocabulary size: 255


In [4]:
# Generate attention masks
train_masks = preprocessor.get_attention_mask(train_sequences)
val_masks = preprocessor.get_attention_mask(val_sequences)

print(f"Train mask shape: {train_masks.shape}")
print(f"Sample mask: {train_masks[0][:20]}...")

Train mask shape: (51000, 64)
Sample mask: [1. 1. 1. 1. 1. 1. 1. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]...


## 2. TF-IDF Features

In [5]:
# Extract TF-IDF features
tfidf_extractor = SequenceFeatureExtractor(
    ngram_range=(1, 3),
    max_features=5000,
    use_tfidf=True,
)

X_train_tfidf = tfidf_extractor.fit_transform(train_sequences)
X_val_tfidf = tfidf_extractor.transform(val_sequences)

print(f"TF-IDF train shape: {X_train_tfidf.shape}")
print(f"TF-IDF val shape: {X_val_tfidf.shape}")

2026-02-28 11:06:54,941 - src.features.sequence_features - INFO - Fitted sequence feature extractor: vocab_size=5000


TF-IDF train shape: (51000, 5000)
TF-IDF val shape: (7200, 5000)


In [6]:
# Check feature sparsity
sparsity = (X_train_tfidf == 0).sum() / X_train_tfidf.size * 100
print(f"Feature matrix sparsity: {sparsity:.2f}%")

Feature matrix sparsity: 99.40%


## 3. Statistical Features

In [7]:
# Extract statistical features
stats_extractor = StatisticalFeatureExtractor(
    include_transitions=True,
    top_k_actions=50,
)

X_train_stats = stats_extractor.fit_transform(train_sequences)
X_val_stats = stats_extractor.transform(val_sequences)

print(f"Statistical train shape: {X_train_stats.shape}")
print(f"Statistical val shape: {X_val_stats.shape}")

2026-02-28 11:06:58,177 - src.features.statistical_features - INFO - Fitted statistical extractor: 50 top actions


Statistical train shape: (51000, 118)
Statistical val shape: (7200, 118)


In [8]:
# Feature names
stat_feature_names = stats_extractor.get_feature_names()
print(f"Number of statistical features: {len(stat_feature_names)}")
print(f"Sample feature names: {stat_feature_names[:10]}")

Number of statistical features: 118
Sample feature names: ['seq_length', 'unique_actions', 'uniqueness_ratio', 'mean_action', 'std_action', 'median_action', 'min_action', 'max_action', 'range_action', 'entropy']


## 4. Complete Feature Pipeline

In [9]:
# Create complete pipeline
pipeline = FeaturePipeline(
    use_tfidf=True,
    use_ngrams=True,
    use_statistics=True,
    ngram_range=(1, 3),
    max_features=5000,
    top_k_actions=50,
)

X_train_features = pipeline.fit_transform(train_sequences)
X_val_features = pipeline.transform(val_sequences)

print(f"Final train features shape: {X_train_features.shape}")
print(f"Final val features shape: {X_val_features.shape}")

2026-02-28 11:07:25,882 - src.features.feature_pipeline - INFO - Fitting feature pipeline on 51000 sequences...
2026-02-28 11:07:25,883 - src.features.feature_pipeline - INFO - Fitting tfidf extractor...
2026-02-28 11:07:27,206 - src.features.sequence_features - INFO - Fitted sequence feature extractor: vocab_size=5000
2026-02-28 11:07:27,206 - src.features.feature_pipeline - INFO - Fitting ngram extractor...
2026-02-28 11:07:29,222 - src.features.sequence_features - INFO - Fitted n-gram extractor: 300 features
2026-02-28 11:07:29,239 - src.features.feature_pipeline - INFO - Fitting stats extractor...
2026-02-28 11:07:29,622 - src.features.statistical_features - INFO - Fitted statistical extractor: 50 top actions
2026-02-28 11:07:29,637 - src.features.feature_pipeline - INFO - Fitting histogram extractor...
2026-02-28 11:07:29,672 - src.features.statistical_features - INFO - Fitted HistogramFeatureExtractor: vocab=254 actions, last_k=5
2026-02-28 11:07:29,672 - src.features.feature_pip

Final train features shape: (51000, 6942)
Final val features shape: (7200, 6942)


In [10]:
# Save pipeline
pipeline.save('../outputs/models/feature_pipeline.pkl')
print("Feature pipeline saved!")

2026-02-28 11:08:11,262 - src.features.feature_pipeline - INFO - Saved feature pipeline to ../outputs/models/feature_pipeline.pkl


Feature pipeline saved!


## 5. Target Encoding

In [11]:
# Encode targets
target_encoder = TargetEncoder(DataLoader.TARGET_COLS)
y_train = target_encoder.fit_transform(Y_train)
y_val = target_encoder.transform(Y_val)

print(f"Encoded y_train shape: {y_train.shape}")
print(f"Encoded y_val shape: {y_val.shape}")
print(f"\nNumber of classes per target: {target_encoder.num_classes}")

2026-02-28 11:08:11,282 - src.data.preprocessor - INFO - Fitted target encoder: {'attr_1': 12, 'attr_2': 31, 'attr_3': 99, 'attr_4': 12, 'attr_5': 31, 'attr_6': 99}


Encoded y_train shape: (51000, 6)
Encoded y_val shape: (7200, 6)

Number of classes per target: {'attr_1': 12, 'attr_2': 31, 'attr_3': 99, 'attr_4': 12, 'attr_5': 31, 'attr_6': 99}


## Summary

### Features Created

| Feature Group | Extractor | # Features | Used by |
|---------------|-----------|-----------|---------|
| TF-IDF (ngram 1–3) | `SequenceFeatureExtractor` | 5,000 | XGBoost, LightGBM |
| N-gram counts | `SequenceFeatureExtractor` | 300 | XGBoost, LightGBM |
| Statistical | `StatisticalFeatureExtractor` | 118 | XGBoost, LightGBM |
| Histogram (254 bins) + last-K | `HistogramFeatureExtractor` | 1,524 | XGBoost, LightGBM |
| **Total ML features** | `FeaturePipeline` | **6,942** | XGBoost, LightGBM |
| Padded sequences (dense vocab) | `SequencePreprocessor` | 64 (length) | LSTM, Transformer |

### Data Leakage Verification

- `SequencePreprocessor.fit()` called on `train_sequences` only
- `FeaturePipeline.fit()` called on `train_sequences` only
- `TargetEncoder.fit()` called on `Y_train` only
- Dense vocab mapping built from train tokens only (254 unique actions → indices 1–254)
- Val/test sequences transformed using pre-fitted preprocessors (no new information leaked)

### Next Steps (Notebook 03)

- Train XGBoost and LightGBM (ML) using `FeaturePipeline` features
- Train LSTM (DL) and Transformer using padded sequences
- Build Ensemble (weighted soft-voting)
- All models trained on `X_train` with **internal early-stop fold only** — `Y_val` is held out
- Evaluate all models on `X_val / Y_val` (held-out) and report Exact-Match Accuracy